# 05 — Transfer Learning: DenseNet121

**Goal of this notebook**: train DenseNet121 (pretrained on ImageNet) in two phases —
frozen base first, then fine-tune the last layers — and compare against the custom CNN
baseline from notebook 04.

**Note on input channels**: DenseNet expects 3-channel input, but our MRI images are
single-channel grayscale (see `build_tf_dataset`, which returns shape `(H, W, 1)`). We
replicate the channel 3x below rather than modifying the shared loader, since the custom
CNN baseline correctly wants 1 channel.

In [ ]:
import sys

sys.path.insert(0, "..")

from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf

from src.data_utils import build_tf_dataset
from src.evaluate import evaluate_predictions
from src.models import build_transfer_model, unfreeze_for_finetuning
from src.train import train_model

CLASS_NAMES = ["glioma", "meningioma", "pituitary", "no_tumor"]
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

In [ ]:
split_metadata = pd.read_csv("../data/processed/metadata_split.csv")


def to_3channel(images, labels):
    return tf.repeat(images, repeats=3, axis=-1), labels


train_dataset = build_tf_dataset(
    split_metadata, "train", CLASS_NAMES, img_size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=True
).map(to_3channel)
val_dataset = build_tf_dataset(
    split_metadata, "val", CLASS_NAMES, img_size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=False
).map(to_3channel)
test_dataset = build_tf_dataset(
    split_metadata, "test", CLASS_NAMES, img_size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=False
).map(to_3channel)

## Data augmentation

**Intent**: our effective per-class training data shrinks after the patient-level split
— augmentation matters more here than it did for the baseline. Applied only to the
training set, inside the model so it's also active for `.fit()` but skipped at
inference automatically (Keras handles this via the `training` argument).

In [ ]:
augmentation = tf.keras.Sequential(
    [
        tf.keras.layers.RandomRotation(0.05),
        tf.keras.layers.RandomZoom(0.1),
        tf.keras.layers.RandomFlip("horizontal"),
    ],
    name="augmentation",
)

train_dataset_augmented = train_dataset.map(
    lambda x, y: (augmentation(x, training=True), y),
    num_parallel_calls=tf.data.AUTOTUNE,
)

## Phase 1 — frozen base

**Intent**: train only the new classification head first, with DenseNet's pretrained
weights frozen — this avoids destroying useful ImageNet features with large, noisy
gradients before the head has learned anything reasonable.

In [ ]:
model = build_transfer_model(
    input_shape=(*IMG_SIZE, 3), num_classes=len(CLASS_NAMES), freeze_base=True
)

history_phase1 = train_model(
    model,
    train_dataset_augmented,
    val_dataset,
    run_name="densenet121_phase1_frozen",
    epochs=15,
    checkpoint_dir=Path("../models/saved_models"),
)

## Phase 2 — fine-tune

**Intent**: unfreeze the last ~30 layers of DenseNet and continue training at a much
lower learning rate, so the model adapts its high-level features to MRI-specific
patterns without wrecking the useful low-level pretrained features.

In [ ]:
model = unfreeze_for_finetuning(model, num_layers_to_unfreeze=30)

history_phase2 = train_model(
    model,
    train_dataset_augmented,
    val_dataset,
    run_name="densenet121_phase2_finetuned",
    epochs=15,
    checkpoint_dir=Path("../models/saved_models"),
)

## Evaluate on the held-out test set

In [ ]:
y_true_idx, y_pred_idx = [], []
for images, labels in test_dataset:
    preds = model.predict(images, verbose=0)
    y_pred_idx.extend(preds.argmax(axis=1))
    y_true_idx.extend(labels.numpy())

y_true = np.array([CLASS_NAMES[i] for i in y_true_idx])
y_pred = np.array([CLASS_NAMES[i] for i in y_pred_idx])

results = evaluate_predictions(y_true, y_pred, CLASS_NAMES)
print(results["report"])
print("\nConfusion matrix:")
print(results["confusion_matrix"])
print(f"\nno_tumor_miss_rate: {results['no_tumor_miss_rate']}")

Next notebook: **06_model_comparison.ipynb** — side-by-side comparison of this
model against the notebook 04 baseline.